# Notebook 03 — Insights e Ações

**Pergunta central:** O que a empresa pode fazer com esses dados?

Tradução dos achados em recomendações concretas para o RH. Cada insight vira uma ação: o que monitorar, quando agir, qual o custo de não agir.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/raw/hr_dashboard_data.csv')
df['Joining Date'] = pd.to_datetime(df['Joining Date'], format='%b-%y')
df['anos_empresa'] = ((pd.Timestamp('2024-01-01') - df['Joining Date']).dt.days / 365).round(1)
df['nivel_satisfacao'] = pd.cut(
    df['Satisfaction Rate (%)'],
    bins=[0, 33, 66, 100],
    labels=['Baixa', 'Média', 'Alta']
)
print('Dados carregados:', len(df), 'colaboradores')

Dados carregados: 200 colaboradores


## 1. Segmentação de Risco — Quem precisa de atenção agora?

In [3]:
# colaboradores com satisfação baixa E produtividade baixa = risco alto
df['risco'] = 'Normal'
df.loc[(df['Satisfaction Rate (%)'] < 33) & (df['Productivity (%)'] < 40), 'risco'] = 'Alto'
df.loc[(df['Satisfaction Rate (%)'] < 33) & (df['Productivity (%)'] >= 40), 'risco'] = 'Médio'
df.loc[(df['Satisfaction Rate (%)'] >= 33) & (df['Productivity (%)'] < 40), 'risco'] = 'Atenção'

fig = px.scatter(
    df,
    x='Satisfaction Rate (%)', y='Productivity (%)',
    color='risco',
    color_discrete_map={
        'Alto': '#ef5350',
        'Médio': '#ffa726',
        'Atenção': '#ffee58',
        'Normal': '#66bb6a'
    },
    title='Mapa de Risco: Satisfação × Produtividade',
    labels={'Satisfaction Rate (%)': 'Satisfação (%)', 'Productivity (%)': 'Produtividade (%)'},
    hover_data=['Name', 'Department', 'Position']
)
fig.add_vline(x=33, line_dash='dash', line_color='gray')
fig.add_hline(y=40, line_dash='dash', line_color='gray')
fig.show()

print('=== SEGMENTAÇÃO DE RISCO ===')
print(df['risco'].value_counts().to_string())
print(f'\nColaboradores em risco alto: {(df["risco"] == "Alto").sum()} ({(df["risco"] == "Alto").mean()*100:.1f}%)')

=== SEGMENTAÇÃO DE RISCO ===
risco
Normal     79
Atenção    58
Médio      34
Alto       29

Colaboradores em risco alto: 29 (14.5%)


## 2. Departamentos Críticos

In [4]:
depto_risco = df.groupby('Department').agg(
    satisfacao_media=('Satisfaction Rate (%)', 'mean'),
    produtividade_media=('Productivity (%)', 'mean'),
    feedback_medio=('Feedback Score', 'mean'),
    total=('Name', 'count'),
    risco_alto=('risco', lambda x: (x == 'Alto').sum())
).reset_index()

depto_risco['pct_risco_alto'] = (depto_risco['risco_alto'] / depto_risco['total'] * 100).round(1)
depto_risco = depto_risco.sort_values('satisfacao_media')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Satisfação Média por Departamento', '% em Risco Alto por Departamento']
)

fig.add_trace(
    go.Bar(
        y=depto_risco['Department'],
        x=depto_risco['satisfacao_media'],
        orientation='h',
        marker_color='#2196F3',
        text=depto_risco['satisfacao_media'].round(1).astype(str) + '%',
        textposition='outside',
        name='Satisfação'
    ), row=1, col=1
)

fig.add_trace(
    go.Bar(
        y=depto_risco['Department'],
        x=depto_risco['pct_risco_alto'],
        orientation='h',
        marker_color='#ef5350',
        text=depto_risco['pct_risco_alto'].astype(str) + '%',
        textposition='outside',
        name='Risco Alto'
    ), row=1, col=2
)

fig.update_layout(title='Diagnóstico por Departamento', showlegend=False, height=400)
fig.show()

print(depto_risco[['Department', 'satisfacao_media', 'produtividade_media', 'feedback_medio', 'pct_risco_alto']].to_string(index=False))

Department  satisfacao_media  produtividade_media  feedback_medio  pct_risco_alto
 Marketing         46.023810            44.261905        3.140476            19.0
     Sales         48.617021            44.212766        2.876596            19.1
   Finance         50.048780            42.268293        2.709756             9.8
        HR         51.625000            48.125000        2.625000            12.5
        IT         54.342105            56.342105        3.010526            10.5


## 3. O Problema do Feedback Baixo

In [5]:
# colaboradores com feedback baixo por departamento
df['feedback_baixo'] = df['Feedback Score'] < 2

fb_depto = df.groupby('Department').agg(
    total=('Name', 'count'),
    feedback_baixo=('feedback_baixo', 'sum'),
    satisfacao_media=('Satisfaction Rate (%)', lambda x: x[df.loc[x.index, 'feedback_baixo']].mean())
).reset_index()

fb_depto['pct_fb_baixo'] = (fb_depto['feedback_baixo'] / fb_depto['total'] * 100).round(1)

fig = px.bar(
    fb_depto.sort_values('pct_fb_baixo', ascending=False),
    x='Department', y='pct_fb_baixo',
    title='% de Colaboradores com Feedback Baixo (< 2.0) por Departamento',
    labels={'Department': 'Departamento', 'pct_fb_baixo': '% com Feedback Baixo'},
    color='pct_fb_baixo',
    color_continuous_scale=['#66bb6a', '#ffa726', '#ef5350'],
    text=fb_depto.sort_values('pct_fb_baixo', ascending=False)['pct_fb_baixo'].astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 4. Tabela de Ações Recomendadas

In [6]:
acoes = pd.DataFrame({
    'Problema': [
        'Satisfação < 33%',
        'Feedback Score < 2.0',
        'Produtividade < 40%',
        'Satisfação cai com tempo de casa',
        'Diferença de satisfação entre departamentos'
    ],
    'Sinal de Alerta': [
        f'{(df["Satisfaction Rate (%)"] < 33).sum()} colaboradores ({(df["Satisfaction Rate (%)"] < 33).mean()*100:.0f}%)',
        f'{(df["Feedback Score"] < 2).sum()} colaboradores ({(df["Feedback Score"] < 2).mean()*100:.0f}%)',
        f'{(df["Productivity (%)"] < 40).sum()} colaboradores ({(df["Productivity (%)"] < 40).mean()*100:.0f}%)',
        'Risco de estagnação após 5 anos',
        'Variação de até 20pp entre áreas'
    ],
    'Ação Recomendada': [
        'Pesquisa de clima imediata + 1:1 com gestores',
        'Capacitar gestores em feedback construtivo',
        'Identificar obstáculos — carga, ferramentas ou motivação',
        'Trilha de carreira clara nos anos 3-5',
        'Investigar práticas dos departamentos mais satisfeitos'
    ],
    'Custo de Não Agir': [
        'Risco de turnover e queda de produtividade',
        'Desmotivação silenciosa — não aparece até ser tarde',
        'Prejuízo direto em entregas e resultados',
        'Turnover de colaboradores experientes (custo alto)',
        'Desengajamento e percepção de injustiça'
    ]
})

print(acoes.to_string(index=False))

                                   Problema                  Sinal de Alerta                                         Ação Recomendada                                   Custo de Não Agir
                           Satisfação < 33%           63 colaboradores (32%)            Pesquisa de clima imediata + 1:1 com gestores          Risco de turnover e queda de produtividade
                       Feedback Score < 2.0           52 colaboradores (26%)               Capacitar gestores em feedback construtivo Desmotivação silenciosa — não aparece até ser tarde
                        Produtividade < 40%           87 colaboradores (44%) Identificar obstáculos — carga, ferramentas ou motivação            Prejuízo direto em entregas e resultados
           Satisfação cai com tempo de casa  Risco de estagnação após 5 anos                    Trilha de carreira clara nos anos 3-5  Turnover de colaboradores experientes (custo alto)
Diferença de satisfação entre departamentos Variação de até 20pp entre

## 5. Resumo Executivo

In [7]:
depto_menor_sat = df.groupby('Department')['Satisfaction Rate (%)'].mean().idxmin()
depto_maior_sat = df.groupby('Department')['Satisfaction Rate (%)'].mean().idxmax()
cargo_menor_sat = df.groupby('Position')['Satisfaction Rate (%)'].mean().idxmin()

print('=== RESUMO EXECUTIVO ===')
print(f'\nTotal analisado: {len(df)} colaboradores')
print(f'Satisfação média: {df["Satisfaction Rate (%)"].mean():.1f}%')
print(f'Produtividade média: {df["Productivity (%)"].mean():.1f}%')
print(f'\nColaboradores em risco alto: {(df["risco"] == "Alto").sum()} ({(df["risco"] == "Alto").mean()*100:.1f}%)')
print(f'Departamento com menor satisfação: {depto_menor_sat}')
print(f'Departamento com maior satisfação: {depto_maior_sat}')
print(f'Cargo com menor satisfação: {cargo_menor_sat}')
print(f'\nCorrelação satisfação × produtividade: {stats.pearsonr(df["Satisfaction Rate (%)"], df["Productivity (%)"])[0]:.3f}')
print(f'Correlação satisfação × feedback: {stats.pearsonr(df["Satisfaction Rate (%)"], df["Feedback Score"])[0]:.3f}')
print(f'Correlação satisfação × salário: {stats.pearsonr(df["Satisfaction Rate (%)"], df["Salary"])[0]:.3f}')

=== RESUMO EXECUTIVO ===

Total analisado: 200 colaboradores
Satisfação média: 49.9%
Produtividade média: 46.8%

Colaboradores em risco alto: 29 (14.5%)
Departamento com menor satisfação: Marketing
Departamento com maior satisfação: IT
Cargo com menor satisfação: Analyst

Correlação satisfação × produtividade: 0.050
Correlação satisfação × feedback: 0.008
Correlação satisfação × salário: -0.018
